# 11. AI + teoria gier: modelowanie przeciwnika w PyTorch

Uczymy małą sieć neuronową przewidywać następny ruch przeciwnika w RPS na podstawie jego historii, a potem wykorzystujemy prognozę do wyboru najlepszej odpowiedzi.

**Założenie organizacyjne:** notebook działa lokalnie i nie wymaga internetu.

In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(7)
torch.manual_seed(7)

# Przeciwnik ma wzorzec: często powtarza ruch sprzed 2 tur, ale z 20% szumu.
def generate_moves(n=5000, seed=7):
    rng=np.random.default_rng(seed)
    moves=[int(rng.integers(3)), int(rng.integers(3)), int(rng.integers(3))]
    for t in range(3,n):
        if rng.random()<0.80:
            nxt=moves[-2]
        else:
            nxt=int(rng.integers(3))
        moves.append(nxt)
    return np.array(moves)

moves=generate_moves()

def features_targets(moves, k=3):
    X=[]; y=[]
    for t in range(k,len(moves)):
        feat=np.zeros((k,3),dtype=np.float32)
        for i,m in enumerate(moves[t-k:t]): feat[i,m]=1.0
        X.append(feat.ravel()); y.append(moves[t])
    return np.array(X,np.float32), np.array(y,np.int64)

X,y=features_targets(moves)
split=int(0.8*len(X))
Xtr,Xte=X[:split],X[split:]; ytr,yte=y[:split],y[split:]
print(Xtr.shape, Xte.shape)

(3997, 9) (1000, 9)


In [2]:
model=nn.Sequential(
    nn.Linear(9,24), nn.ReLU(),
    nn.Linear(24,3)
)
opt=torch.optim.Adam(model.parameters(),lr=0.01)
loss_fn=nn.CrossEntropyLoss()
loader=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=128,shuffle=True)

for epoch in range(12):
    model.train(); total=0.0
    for xb,yb in loader:
        opt.zero_grad(); logits=model(xb); loss=loss_fn(logits,yb)
        loss.backward(); opt.step(); total += loss.item()*len(xb)
    if epoch in [0,3,7,11]:
        print('epoch',epoch,'loss',round(total/len(Xtr),4))

model.eval()
with torch.no_grad():
    pred=model(torch.tensor(Xte)).argmax(1).numpy()
acc=(pred==yte).mean()
print('test accuracy=',round(acc,3))

epoch 0 loss 0.7477


epoch 3 loss 0.494


epoch 7 loss 0.4944


epoch 11 loss 0.494
test accuracy= 0.856


In [3]:
# Best response: 0=Rock,1=Paper,2=Scissors. Odpowiedź wygrywająca to (ruch+1)%3.
br=(pred+1)%3
# wypłata: +1 wygrana, 0 remis, -1 przegrana
def rps_reward(our, opp):
    if our==opp: return 0
    return 1 if (our-opp)%3==1 else -1

rewards=np.array([rps_reward(a,b) for a,b in zip(br,yte)])

rng=np.random.default_rng(123)
random_actions=rng.integers(0,3,size=len(yte))
random_rewards=np.array([rps_reward(a,b) for a,b in zip(random_actions,yte)])

print('Średnia nagroda z opponent model:',round(rewards.mean(),3))
print('Średnia nagroda losowo:',round(random_rewards.mean(),3))

Średnia nagroda z opponent model: 0.794
Średnia nagroda losowo: -0.026


## Zadanie
1. Zwiększ szum przeciwnika z 20% do 40%.
2. Zmień jego regułę zachowania, np. „najczęściej kontruje nasz ostatni ruch”.
3. Sprawdź, czy wysoka accuracy zawsze daje wysoką wypłatę strategiczną.

**Wniosek:** metryka predykcyjna i metryka decyzyjna to dwie różne rzeczy. Model przeciwnika ma wartość wtedy, gdy poprawia decyzję w grze.